In [1]:
import wave, math, struct
import tensorflow as tf
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, Activation,Input
# prepare a dumy data for simulating musical notes
notes_freqs={
    'A':440.0, 'B':493.88, 'C':261.63, 'D':293.66, 'E':393.63, 'F':349.23, 'G':392.0
}
notes_freqs
notes = list(notes_freqs.keys())
notes
note_to_int={note  : i for i, note in enumerate(notes)}
int_to_note = {i:note for i, note in enumerate(notes)}
int_to_note
raw_music_data = [notes[np.random.randint(0,7)] for i in range(1000)]
raw_music_data
seq_length=3
network_input=[]
network_output= []

for i in range(len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i: i+seq_length]
    seq_out= raw_music_data[i+seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in, '---->',seq_out)
n_pattern = len(network_input)
n_pattern
x=np.reshape(network_input,(n_pattern,seq_length,1))
from keras.utils import to_categorical
y=to_categorical(network_output)
y.shape

['B', 'B', 'B'] ----> E
['B', 'B', 'E'] ----> A
['B', 'E', 'A'] ----> G
['E', 'A', 'G'] ----> A
['A', 'G', 'A'] ----> B
['G', 'A', 'B'] ----> C
['A', 'B', 'C'] ----> G
['B', 'C', 'G'] ----> C
['C', 'G', 'C'] ----> C
['G', 'C', 'C'] ----> A
['C', 'C', 'A'] ----> F
['C', 'A', 'F'] ----> G
['A', 'F', 'G'] ----> B
['F', 'G', 'B'] ----> B
['G', 'B', 'B'] ----> D
['B', 'B', 'D'] ----> B
['B', 'D', 'B'] ----> F
['D', 'B', 'F'] ----> G
['B', 'F', 'G'] ----> B
['F', 'G', 'B'] ----> D
['G', 'B', 'D'] ----> F
['B', 'D', 'F'] ----> D
['D', 'F', 'D'] ----> B
['F', 'D', 'B'] ----> B
['D', 'B', 'B'] ----> A
['B', 'B', 'A'] ----> C
['B', 'A', 'C'] ----> D
['A', 'C', 'D'] ----> B
['C', 'D', 'B'] ----> A
['D', 'B', 'A'] ----> D
['B', 'A', 'D'] ----> F
['A', 'D', 'F'] ----> A
['D', 'F', 'A'] ----> C
['F', 'A', 'C'] ----> C
['A', 'C', 'C'] ----> G
['C', 'C', 'G'] ----> D
['C', 'G', 'D'] ----> E
['G', 'D', 'E'] ----> A
['D', 'E', 'A'] ----> A
['E', 'A', 'A'] ----> F
['A', 'A', 'F'] ----> A
['A', 'F', 'A'] 

(997, 7)

In [2]:
model=Sequential()
model.add(Input((3,1)))
model.add(GRU(256))
model.add(Dense(512,activation='relu'))
model.add(Dense(7,activation='softmax'))


In [3]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 256)            │       198,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,087 (1.27 MB)

 Trainable params: 334,087 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
%pip install flax

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.7 MB 6.3 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/57.9 MB ? eta -:--:--
    --------------------------------------- 0.8/57.9 MB 5.6 MB/s eta 0:00:11
   - -------------------------------------- 2.1/57.9 MB 5.3 MB/s eta 0:00:11
   -- ------------------------------------- 3.1/57.9 MB 5.4 MB/s eta 0:00:11
   --- ------------------------------------ 4.5/57.9 MB 5.4 MB/s eta 0:00:10
   --- ------------------------------------ 5.2/57.9 MB 5.1 MB/s eta 0:00:11
   ---- ----------------------------------- 6.0/57.9 MB 4.8 MB/s eta 0:00:11
   ---- ----------------------------------- 7.1/57.9 MB 4.8 MB/s eta 0:00:11
   ----- ---------------------------------- 7.9/57.9 MB 4.8 MB/s eta 0:00:11
   ------ ----------


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from keras.src.metrics import metric
from flax.nnx import optimizer
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [7]:
model.fit(x,y,epochs=1000,batch_size=10)

Epoch 1/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1314 - loss: 1.9739
Epoch 2/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1494 - loss: 1.9544
Epoch 3/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.1384 - loss: 1.9502
Epoch 4/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1525 - loss: 1.9471
Epoch 5/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1525 - loss: 1.9439
Epoch 6/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1384 - loss: 1.9453
Epoch 7/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1384 - loss: 1.9454
Epoch 8/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1565 - loss: 1.9421
Epoch 9/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1595 - loss: 1.9401
Epoch 10/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1545 - loss: 1.9393
Epoch 11/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1585 - loss: 1.9430
Epoch 12/1000
100/100 ━━━━━━━━

In [8]:
# generate new melody seq
start_index= np.random.randint(0, len(network_output))
pattern = network_input[start_index]
pattern


[6, 3, 1]

In [9]:
generated_melody=[]
for i in range(50):
  x_input = np.reshape(pattern,(1,len(pattern),1))
  pred = model.predict(x_input, verbose=0)
  index = np.argmax(pred)
  result = int_to_note[index]
  generated_melody.append(result)
  pattern.append(index)
  pattern=pattern[1:len(pattern)]

In [10]:
generated_melody

['E',
 'B',
 'A',
 'D',
 'D',
 'C',
 'A',
 'C',
 'G',
 'G',
 'E',
 'C',
 'G',
 'D',
 'E',
 'G',
 'C',
 'D',
 'C',
 'A',
 'C',
 'G',
 'G',
 'E',
 'C',
 'G',
 'D',
 'E',
 'G',
 'C',
 'D',
 'C',
 'A',
 'C',
 'G',
 'G',
 'E',
 'C',
 'G',
 'D',
 'E',
 'G',
 'C',
 'D',
 'C',
 'A',
 'C',
 'G',
 'G',
 'E']

In [ ]:
# with wave.open('my_music.wav','w') as wav_file:
#   wav_file.setparams((1,2,44100,0,'NONE','not compressed'))
#   for note in generated_melody:
#     freq = notes_freqs[note]
#     num_samples=int(0.5 * 44100)
#     t=float(i)/44100
#     value= int(32767 * 0.5 * math.sin(2 * math.pi * freq * t))
#     data = struct.pack('<h',value)
#     wav_file.writeframes(data)

In [12]:
with wave.open('my_music.wav','w') as wav_file:
    wav_file.setparams((1, 2, 44100, 0, 'NONE', 'not compressed'))

    sample_rate = 44100
    duration = 0.5

    for note in generated_melody:
        freq = notes_freqs[note]
        num_samples = int(duration * sample_rate)

        for i in range(num_samples):
            t = i / sample_rate
            value = int(32767 * 0.5 * math.sin(2 * math.pi * freq * t))
            wav_file.writeframes(struct.pack('<h', value))